In [1]:
import sys
import maboss
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append("/Users/emilieyu/endotehelial-masboss")
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, RegularGridInterpolator

from src.config import load_sim_config, load_sweep_config, load_spatial_config
from src.paths import *
from boolean_models.scripts import run_perturbations, run_sweeps

sim_cfg = load_sim_config()
sweep_cfg = load_sweep_config()
spatial_cfg = load_spatial_config()

MODELS_BND = DATA_DIR / sim_cfg['model']['bnd_v2']
MODELS_CFG = DATA_DIR / sim_cfg['model']['cfg_v4']

from abm.flow_field import FlowField
from abm.endothelial_cell import EndothelialCell
from abm.membrane_node import MembraneNode
from abm.rho_lookup_table import RhoLookupTable
from abm.spring import Spring

ipylab module is not installed, menus and toolbar are disabled.


## Test Springs Only

In [2]:
node1 = MembraneNode(0, np.array([0, 1]))
node2 = MembraneNode(1, np.array([1, 3]))
lut = RhoLookupTable(spatial_cfg, BM_RESULTS_DIR)

spring = Spring(0, node1, node2, 1, lut, spatial_cfg)
spring



>>> DEBUG: Successfully loaded recruitment parameter sweep data.


Spring(0→1 | L=1.000 L0=1.000 | T=0.000 align=0.000 | DSP=0.0 TJP1=0.0 JCAD=0.0 | RhoA=0.000 RhoC=0.000)

In [3]:
spring.update_geometry(np.array([1, 0]))
spring

>>>DEBUG: diff: [1. 2.] | length: 2.23606797749979 | unit vect: [0.4472136  0.89442719]
              alignment: 0.4472135954999579


Spring(0→1 | L=2.236 L0=1.000 | T=1.236 align=0.447 | DSP=0.0 TJP1=0.0 JCAD=0.0 | RhoA=0.000 RhoC=0.000)

In [4]:
spring.update_signalling()
spring

>>> DEBUG: Computed mechnical input tau_dsp: 0.5527864045000421, tau_tjp1: 0.6832815729997478, tau_jcad: 1.2360679774997898
>>>INFO: Protein Recruitment for spring 0 is
               DSP: 0.04766379079954211, TJP1: 0.08635778723311313, JCAD: 0.3587970167215063
>>>INFO: Rho Activation for sprinf 0 is 
              RhoA: 0.5573891767518151, RhoC: 0.57262566265475


Spring(0→1 | L=2.236 L0=1.000 | T=1.236 align=0.447 | DSP=0.04766379079954211 TJP1=0.08635778723311313 JCAD=0.3587970167215063 | RhoA=0.557 RhoC=0.573)

In [5]:
spring.update_stiffness()
spring

k_target: 2.6721675302554453
l_shrink_rhoa: 0.11147783535036303, l_shrink_rhoc: 0.12804299073568837
L_target: {L_target}
k_cortical: 1.0083608376512772, L_cortical: 0.9988023958695698


Spring(0→1 | L=2.236 L0=1.000 | T=1.236 align=0.447 | DSP=0.04766379079954211 TJP1=0.08635778723311313 JCAD=0.3587970167215063 | RhoA=0.557 RhoC=0.573)

## Recruitment Parameter Table

In [2]:
LUT = RhoLookupTable(spatial_cfg, BM_RESULTS_DIR)


>>> DEBUG: Successfully loaded recruitment parameter sweep data.
>>> DEBUG: Cleaned recruitment parameter sweep data.
>>> DEBUG: Successfully built interpolators
LUT ready | rest: RhoA=0.463 RhoC=0.437


In [7]:
LUT.query(0.0, 1.0, 0.0)

(0.367, 0.606)

## FLOW TO MECHNICAL INPUT FOR MABOSS

In [1]:
lut = RhoLookupTable(spatial_cfg, BM_RESULTS_DIR)
cell = EndothelialCell(0, np.array([0.0, 0.0]))
field = FlowField()

print("Initial Node State: ")
for node in cell.nodes: 
    print(node)

print("Initial Sping State: ")
for spring in cell.springs: 
    print(spring)

NameError: name 'RhoLookupTable' is not defined

In [5]:
flow_force = field.get_force_on_node()

print("Nodes after applying force from flow")
for node in cell.nodes: 
    node.apply_force(flow_force)
    print(node)

print("Spring after updating geometry")
for spring in cell.springs: 
    spring.update_geometry(field.direction)
    print(spring)

Nodes after applying force from flow
MembraneNode(id=0 | pos=[10.  0.] | force=[1. 0.]
MembraneNode(id=1 | pos=[8.66 5.  ] | force=[1. 0.]
MembraneNode(id=2 | pos=[5.   8.66] | force=[1. 0.]
MembraneNode(id=3 | pos=[ 0. 10.] | force=[1. 0.]
MembraneNode(id=4 | pos=[-5.    8.66] | force=[1. 0.]
MembraneNode(id=5 | pos=[-8.66  5.  ] | force=[1. 0.]
MembraneNode(id=6 | pos=[-10.   0.] | force=[1. 0.]
MembraneNode(id=7 | pos=[-8.66 -5.  ] | force=[1. 0.]
MembraneNode(id=8 | pos=[-5.   -8.66] | force=[1. 0.]
MembraneNode(id=9 | pos=[ -0. -10.] | force=[1. 0.]
MembraneNode(id=10 | pos=[ 5.   -8.66] | force=[1. 0.]
MembraneNode(id=11 | pos=[ 8.66 -5.  ] | force=[1. 0.]
Spring after updating geometry
Spring(0→1 | L=5.176 L0=5.176 | T=-0.000 align=0.259 | DSP=0 TJP1=0 JCAD=0 | RhoA=0.000 RhoC=0.000)
Spring(1→2 | L=5.176 L0=5.176 | T=0.000 align=0.707 | DSP=0 TJP1=0 JCAD=0 | RhoA=0.000 RhoC=0.000)
Spring(2→3 | L=5.176 L0=5.176 | T=0.000 align=0.966 | DSP=0 TJP1=0 JCAD=0 | RhoA=0.000 RhoC=0.000)


In [6]:
spring2 = cell.springs[2]
spring2

Spring(2→3 | L=5.176 L0=5.176 | T=0.000 align=0.966 | DSP=0 TJP1=0 JCAD=0 | RhoA=0.000 RhoC=0.000)

In [7]:

p_dsp, p_tjp1, p_jcad = spring2.update_signalling(spatial_cfg)
p_dsp, p_tjp1, p_jcad

>>> DEBUG: Computed mechnical input tau_dsp: 8.579144739409623e-16, tau_tjp1: 3.0263945759162957e-17, tau_jcad: 8.881784197001252e-16


(np.float64(1.8709328806229727e-46),
 np.float64(8.213019852114197e-51),
 np.float64(2.0759977249256548e-46))

In [8]:

lut.query(p_dsp, p_tjp1, p_jcad)

(np.float64(0.5243333333333333), np.float64(0.5286666666666667))